# Train and inspect SurvivalMDN on SUPPORT

This notebook shows how to train a SurvivalMDN model on the SUPPORT survival
benchmark, load a saved model, plot predicted survival curves, and verify that
reloading a checkpoint reproduces the predictions. It uses a fixed PyHealth
baseline and the prepared training, validation, and test splits distributed
with the Survival-MDN repository.

The companion [script](survival_mdn_support.py) handles data loading, training,
and prediction.

This notebook uses preprocessed SUPPORT data and a fixed training configuration. See the [SUPPORT guide](survival_mdn_support_README.md)
for fingerprints, preprocessing audit, frozen settings, and reference
architecture, numerical, optimizer, and sampler differences.

## Setup

Use Python 3.12 or 3.13 with this checkout installed in editable mode. From the
repository root:

```bash
python -m pip install -e .
python -m pip install jupyterlab ipykernel matplotlib pandas
python -m ipykernel install --user --name pyhealth-survival --display-name "PyHealth Survival"
python -m jupyterlab
```

Select the **PyHealth Survival** kernel. Obtain the archives locally from the
[SUPPORT data in the Survival-MDN repository](https://github.com/XintianHan/Survival-MDN/tree/71c61f649f3c8784fe13b092b38e0cbb3a184dbd/data/support).
Nothing in this notebook downloads or redistributes data.

The default inspects an existing `output/support-baseline` run. To train the
frozen baseline, set `RUN_TRAINING = True` below; it chooses a fresh directory.
Each replay also uses a fresh directory. Rerunning setup chooses new names;
rerunning a training/replay cell alone refuses to overwrite its previous run.
Keep clinical records, curves, weights, and executed notebook outputs local.
Clear all cell outputs before committing or sharing this notebook.


In [ ]:
import importlib.util
import json
import re
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import pyhealth
from pyhealth.datasets import get_dataloader
from pyhealth.models import SurvivalMDN

candidates = [p for base in (Path.cwd(), *Path.cwd().parents)
              for p in (base, base / "PyHealth")]
repo = next((p for p in candidates
             if (p / "examples/survival_mdn_support.py").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the PyHealth checkout.")
if not Path(pyhealth.__file__).resolve().is_relative_to(repo.resolve()):
    raise RuntimeError("Select a kernel with this checkout installed using pip install -e .")

spec = importlib.util.spec_from_file_location(
    "support_demo", repo / "examples/survival_mdn_support.py")
support = importlib.util.module_from_spec(spec)
spec.loader.exec_module(support)  # main guard prevents training on import.

# Edit these local paths for your environment; no downloads or random splits.
DATA_DIR = repo.parent / "Survival-MDN" / "data" / "support"
EXISTING_RUN_DIR = repo / "output" / "support-baseline"
RUN_TRAINING = False
session_id = uuid4().hex[:8]
TRAIN_RUN_DIR = repo / "output" / f"support-notebook-baseline-{session_id}"
REPLAY_DIR = repo / "output" / f"support-notebook-replay-{session_id}"
RUN_DIR = TRAIN_RUN_DIR if RUN_TRAINING else EXISTING_RUN_DIR
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Set DATA_DIR to your local released SUPPORT directory.")
print("CPU float32 baseline; training enabled:", RUN_TRAINING)


## 1. Inspect the released partitions

The pinned source revision is `71c61f649f3c8784fe13b092b38e0cbb3a184dbd`.
The adapter preserves train/valid/test archive row order and all 27 columns.
`arr_0` provides features; `arr_1` provides duration/event labels. The additional
`arr_2` array is ignored. Training-column centering/scaling of `arr_2`
approximates `arr_0`, but the archive-generation history, feature names,
imputation choices, and cohort filtering are not verified.

The table contains aggregate statistics and hashes only. Compare the hashes to
the pinned split-1 fingerprints in the guide before calling a run the pinned
benchmark. Record IDs identify archive rows, not verified original patient IDs;
all archive rows are retained.


In [ ]:
loaded = support.load_support_split(DATA_DIR, split=1)
datasets, metadata = loaded["datasets"], loaded["metadata"]
rows = []
for phase, audit in metadata["partitions"].items():
    rows.append({
        "partition": phase, "rows": audit["count"],
        "events": audit["events"], "censored": audit["censored"],
        "feature_shape": tuple(audit["feature_shape"]),
        "feature_dtype": audit["feature_dtype"], "label_dtype": audit["label_dtype"],
        "released_duration_min": audit["released_duration_range"][0],
        "released_duration_max": audit["released_duration_range"][1],
        "sha256": audit["sha256"],
    })
with pd.option_context("display.max_colwidth", 70):
    display(pd.DataFrame(rows).set_index("partition"))
print("Calendar-unit mapping:", metadata["original_calendar_unit_mapping"])
print("All selected archive values are finite:", all(
    a["all_selected_values_finite"] for a in metadata["partitions"].values()))


## 2. Understand the model contract and time convention

The input schema is `{"features": "tensor"}` and outputs are
`{"duration": "tensor", "event": "tensor"}`. A batch has features `[B, 27]` and
scalar labels `[B]`; `event=1` means observed and `event=0` means right censored.
Training tensor processors are reused for held-out partitions and fit no
normalization. Row index, partition, and record ID are metadata, never features.

The adapter computes `(arr_1[:, 0] + 0.001).astype(float32)` once, adding in the
archive's floating precision before conversion. Reloading and prediction add
nothing further. Times use **released SUPPORT time units**, with modeled
horizons shifted by `+0.001`. There is no verified mapping to calendar days.
To query a raw released horizon `h`, query the model at `h + 0.001` once.
Constant held-out feature columns are preserved.


In [ ]:
train = datasets["train"]
shape_batch = next(iter(get_dataloader(train, batch_size=4, shuffle=False)))
display(pd.DataFrame([
    {"field": key, "batch_shape": tuple(shape_batch[key].shape),
     "dtype": str(shape_batch[key].dtype)}
    for key in ("features", "duration", "event")
]))
for phase in ("valid", "test"):
    assert datasets[phase].input_processors["features"] is train.input_processors["features"]
    for key in ("duration", "event"):
        assert datasets[phase].output_processors[key] is train.output_processors[key]
contract_model = SurvivalMDN(train, **support.MODEL_CONFIG).cpu().float()
assert contract_model.mode is None
print("mode=None; held-out partitions reuse training tensor processors.")
del shape_batch, contract_model


## 3. Run or load the frozen baseline

The frozen configuration uses split 1, seed 0, CPU/float32, hidden dimensions
`[64, 64, 64]`, five components, PReLU, no BatchNorm, and minimum scale `1e-4`.
Adam uses learning rate `0.001`, no weight decay, batch size 256, gradient norm
cap 100, up to 100 epochs, and patience 10. Every training row is retained.

The adapter seeds Python/NumPy/PyTorch/the training loader, enables deterministic
algorithms, and uses one intra-op CPU thread. Validation is one full batch,
so Trainer's batch-mean loss is exactly a row mean. Only validation loss selects
weights; test labels never enter training or prediction calls. See the guide for the model's initialization, parameter heads, scale floor,
and likelihood calculations.

With `RUN_TRAINING=False`, this section reads a saved run without training.
The adapter checks an empty output directory before starting a new run. This
notebook checks the saved run against the supplied data and frozen settings.
Exact numeric agreement requires the same software and hardware environment.


In [ ]:
if RUN_TRAINING:
    support.main([
        "--data-dir", str(DATA_DIR), "--split", "1", "--seed", "0",
        "--epochs", "100", "--patience", "10", "--output-dir", str(RUN_DIR),
    ])
manifest_path = RUN_DIR / "manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError(
        "Set EXISTING_RUN_DIR to a completed baseline or enable RUN_TRAINING in setup.")
manifest = json.loads(manifest_path.read_text())
support._check_compatible(metadata, manifest["data"])
config = manifest["config"]
assert manifest["device"] == "cpu" and manifest["dtype"] == "float32"
assert config["model"] == support.MODEL_CONFIG
assert config["seed"] == 0 and metadata["split"] == 1
assert config["maximum_epochs"] == 100 and config["patience"] == 10
print("Artifacts:", RUN_DIR)
display(pd.Series(config, name="Saved configuration").to_frame())
display(pd.Series(manifest["environment"], name="Training environment").to_frame())


## 4. Review validation selection

The final validation NLL below is measured **after best-checkpoint restoration**.
A continuous-time density can exceed one, so negative NLL would not itself
indicate an error.

If a training log is present, plot its validation history. Trainer logs rounded
NLLs; the manifest retains the restored loss at full recorded precision. Mark
the selected epoch from the last actual `New best` log entry, not from the
minimum rounded number or requested epoch budget. Epochs on this plot are
one-based for readability. A prediction-only copy may contain no training log.


In [ ]:
display(pd.Series({
    "Initial training NLL": manifest["initial_training_nll"],
    "Initial validation NLL": manifest["initial_validation_nll"],
    "Restored validation NLL": manifest["restored_validation_nll"],
    "Training seconds": manifest["training_seconds"],
}, name="Saved training summary").to_frame())
log_path = RUN_DIR / "checkpoints" / "log.txt"
if log_path.is_file():
    log_text = log_path.read_text()
    history = re.findall(
        r"--- Eval epoch-(\d+), step-\d+ ---\n[^\n]*loss: ([^\s]+)", log_text)
    selected = re.findall(r"New best loss score .* at epoch-(\d+), step-\d+", log_text)
    if history and selected:
        epochs = np.array([int(e) + 1 for e, loss in history])
        losses = np.array([float(loss) for e, loss in history])
        selected_epoch = int(selected[-1]) + 1
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.plot(epochs, losses, marker=".", label="Validation NLL (rounded log)")
        ax.axvline(selected_epoch, color="tab:orange", linestyle="--",
                   label=f"Selected epoch {selected_epoch}")
        ax.set(xlabel="Epoch (one-based)", ylabel="Validation NLL",
               title="Validation-only checkpoint selection")
        ax.legend()
        fig.tight_layout()
        plt.show()
    else:
        print("No parseable selection history; use the restored NLL above.")
else:
    print("Training log unavailable in this run; no epoch number inferred.")


## 5. Validate and inspect exported curves

Predictions are evaluated in batches of at most 256 with features only, in
eval/no-grad mode. The 128-point float32 grid runs from zero to the largest
transformed **training** duration.

Load exports with `allow_pickle=False`. Check shape, row identity/order,
probability bounds, monotonicity within `1e-6`, and exact ones at time zero.
Duration and event labels accompany each curve; model selection uses validation
data only.
The aggregate plot summarizes predicted curves across test rows. Its percentile
band describes variation among row predictions, not a confidence interval,
calibration, or evidence of predictive accuracy.


In [ ]:
with np.load(RUN_DIR / "test_predictions.npz", allow_pickle=False) as archive:
    predictions = {key: archive[key] for key in archive.files}
survival, times = predictions["survival"], predictions["times"]
count = len(datasets["test"])
assert survival.shape == (count, 128) and times.shape == (128,)
assert times.dtype == np.float32 and times[0] == 0
np.testing.assert_array_equal(times, np.asarray(manifest["prediction_grid"], dtype=np.float32))
assert times[-1] == max(datasets["train"][i]["duration"].item()
                        for i in range(len(datasets["train"])))
np.testing.assert_array_equal(predictions["source_row_index"], np.arange(count))
np.testing.assert_array_equal(predictions["record_id"],
                              [datasets["test"][i]["record_id"] for i in range(count)])
for field in ("duration", "released_duration", "event"):
    expected = [datasets["test"][i][field] for i in range(count)]
    np.testing.assert_array_equal(predictions[field], np.asarray(expected))
assert (predictions["partition"] == "test").all()
assert np.isfinite(survival).all() and ((0 <= survival) & (survival <= 1)).all()
assert (np.diff(survival, axis=1) <= 1e-6).all() and (survival[:, 0] == 1).all()
print(f"Validated all {count} test rows on the saved {len(times)}-point grid.")

lower, upper = np.quantile(survival, [0.1, 0.9], axis=0)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(times, survival.mean(axis=0), label="Mean predicted survival")
ax.fill_between(times, lower, upper, alpha=0.2, label="10th–90th percentile of row curves")
ax.set(xlabel="Model horizon (released SUPPORT time units; +0.001 offset)",
       ylabel="Predicted survival probability", ylim=(0, 1.02),
       title="Predicted survival across held-out archive rows")
ax.legend()
fig.tight_layout()
plt.show()


### Optional: inspect individual archive rows locally

Leave `ROW_INDICES` empty to keep individual curves out of outputs. For local
inspection, set it to a few source row indices, such as `[0, 1, 2]`. These are
archive-row identities, not verified patient identities. The curves are model
predictions for the selected archive rows.
Clear outputs before sharing.


In [ ]:
ROW_INDICES = []
if ROW_INDICES:
    if any(not isinstance(i, int) or not 0 <= i < count for i in ROW_INDICES):
        raise ValueError("ROW_INDICES must contain valid integer test archive row indices.")
    fig, ax = plt.subplots(figsize=(7, 4))
    for row in ROW_INDICES:
        ax.plot(times, survival[row], label=f"Archive row {row}")
    ax.set(xlabel="Model horizon (released SUPPORT time units; +0.001 offset)",
           ylabel="Predicted survival probability", ylim=(0, 1.02),
           title="Optional local inspection of individual predictions")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("Individual-row plots disabled.")


## 6. Verify prediction-only replay

The adapter reconstructs a fresh CPU/float32 model from the saved configuration
and selected state dictionary, validates source hashes and feature width, and
exports predictions without fitting. It reuses the saved grid and does not
apply the duration offset twice. Within each run it also compares all test
predictions against another fresh-model checkpoint reload.

Below, independently compare **every** exported array and checkpoint tensor
against the original run. This confirms replay/row alignment for this
environment. Artifacts stay
in ignored `output/` directories. For an independent training repeat, enable
training in setup to obtain a new run, then compare it with your earlier run.


In [ ]:
support.main([
    "--data-dir", str(DATA_DIR), "--predict-only", str(RUN_DIR),
    "--output-dir", str(REPLAY_DIR),
])
with np.load(REPLAY_DIR / "test_predictions.npz", allow_pickle=False) as replay:
    assert set(replay.files) == set(predictions)
    for key, original in predictions.items():
        np.testing.assert_array_equal(original, replay[key])
original_weights = torch.load(RUN_DIR / manifest["checkpoint"],
                              map_location="cpu", weights_only=True)
replay_weights = torch.load(REPLAY_DIR / manifest["checkpoint"],
                            map_location="cpu", weights_only=True)
assert original_weights.keys() == replay_weights.keys()
for key in original_weights:
    torch.testing.assert_close(original_weights[key], replay_weights[key], atol=0, rtol=0)
print(f"Exact replay: {len(predictions)} arrays, {len(original_weights)} tensors, {count} test rows.")
print("Replay artifacts:", REPLAY_DIR)


## Further reading and checks

- [SUPPORT guide](survival_mdn_support_README.md): pinned data audit, units,
  baseline configuration, exact commands, reference differences, and recorded
  repeat/reload acceptance results.
- [Synthetic tutorial](survival_mdn_synthetic.ipynb): a self-contained introduction
  requiring no clinical archives.
- `tests/core/test_survival_mdn_support.py`: offline generated-archive tests.
- `pyhealth/models/survival_mdn.py`: model and stable likelihood implementation.

Clear every output before committing this notebook.
